In [16]:
import importlib
import src.analytics.pci_calculator as pc

importlib.reload(pc)

print(dir(pc))

['PCICalculator', 'VIOLATION_WEIGHTS', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'pd']


In [17]:
processed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

df = pd.read_pickle(
    processed_path
    / "parking_violations.pkl"
)

print(df.shape)

(298450, 29)


In [18]:
hotspots = pd.read_pickle(
    processed_path
    / "hotspots.pkl"
)

hotspot_df = pd.read_pickle(
    processed_path
    / "hotspot_df.pkl"
)

In [19]:
df["road_criticality"] = (
    df["location"]
    .apply(
        RoadCriticalityCalculator.compute
    )
)

df["road_criticality"].describe()

count    298450.000000
mean          0.720254
std           0.218280
min           0.500000
25%           0.500000
50%           0.750000
75%           1.000000
max           1.000000
Name: road_criticality, dtype: float64

In [20]:
hotspot_sizes = (
    hotspot_df["hotspot_id"]
    .value_counts()
)

hotspot_scores = (
    hotspot_sizes
    / hotspot_sizes.max()
)

df["hotspot_score"] = (
    hotspot_df["hotspot_id"]
    .map(hotspot_scores)
)

df["hotspot_score"] = (
    df["hotspot_score"]
    .fillna(0)
)

In [22]:
import importlib
import src.analytics.pci_calculator as pc

importlib.reload(pc)

PCICalculator = pc.PCICalculator

print(PCICalculator.temporal_score(8))

1.0


In [23]:
df["temporal_score"] = (
    df["hour"]
    .apply(
        PCICalculator.temporal_score
    )
)

df["temporal_score"].describe()

count    298450.000000
mean          0.600434
std           0.282498
min           0.400000
25%           0.400000
50%           0.400000
75%           1.000000
max           1.000000
Name: temporal_score, dtype: float64

In [24]:
df["violation_score"] = (
    df["violation_type"]
    .apply(
        PCICalculator.violation_score
    )
)

df["violation_score"].describe()

count    298450.000000
mean          0.597452
std           0.137284
min           0.500000
25%           0.500000
50%           0.600000
75%           0.600000
max           1.000000
Name: violation_score, dtype: float64

In [25]:
df["pci"] = (
    df.apply(
        PCICalculator.compute_pci,
        axis=1
    )
)

df["pci"].describe()

count    298450.000000
mean          0.529910
std           0.135402
min           0.335000
25%           0.433000
50%           0.489000
75%           0.614000
max           1.000000
Name: pci, dtype: float64

In [26]:
top_locations = (

    df.groupby("location")

    .agg(
        avg_pci=("pci", "mean"),
        violations=("id", "count")
    )

    .sort_values(
        "avg_pci",
        ascending=False
    )
)

top_locations.head(20)

,avg_pci,violations
location,,
"Victoria Hospital Road, Chanda Nagar, Sultanpete, Bengaluru, Karnataka. Pin-560053 (India)",0.990000,1
"B.V.K. Iyengar Road, Balepet, Chikkapete, K.R Market, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560053, India",0.891552,29
"K.R Market, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560053, India",0.890000,1
"Gundopanth Street, K.R Market, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560002, India",0.890000,1
"6th Cross Road, Sultanpet Circle, Chickpete, Bengaluru, Karnataka. Pin-560053 (India)",0.885000,2
"R Subbanna Circle, Sheshadri Road, Gandhinagar, Nehru Nagar, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560009, India",0.885000,2
"Ravi Thivari Road, Sultanpet Circle, Chickpete, Bengaluru, Karnataka. Pin-560053 (India)",0.880385,13
"Kanakadasa Circle, 2nd Cross Road, Gandhinagar, Nehru Nagar, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru Urban, Karnataka, 560009, India",0.860000,1
"4th Main Road, Kempe Gowda Circle, Gandhi Nagar, Bengaluru, Karnataka. Pin-560009 (India)",0.860000,1


In [27]:
df.to_pickle(
    processed_path
    / "pci_scores.pkl"
)

print("PCI dataset saved.")

PCI dataset saved.


In [29]:
df["hotspot_id"] = hotspot_df["hotspot_id"].values

In [30]:
df[[
    "id",
    "latitude",
    "longitude",
    "location",
    "hotspot_id",
    "road_criticality",
    "hotspot_score",
    "temporal_score",
    "violation_score",
    "pci"
]].head()

,id,latitude,longitude,location,hotspot_id,road_criticality,hotspot_score,temporal_score,violation_score,pci
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",0,0.8,0.010453,0.4,1.0,0.573
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",-1,0.8,0.341001,0.4,0.5,0.497
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",0,0.5,0.010453,0.4,1.0,0.513
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",1,0.5,0.008764,1.0,0.5,0.428
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",2,0.5,1.000000,0.4,0.5,0.635


In [31]:
df.to_pickle(
    processed_path
    / "final_pci_dataset.pkl"
)

print("Final dataset saved.")

Final dataset saved.
